# Lab A. Runge's Phenomenon and Chebyshev Nodes

**How this lab is graded.** Completion. Your instructor runs the whole notebook
(Runtime > Run all) and checks that every cell produces output and every self-check
prints PASS. No written answers are submitted here; the questions about what these
experiments mean, and the Part 2 derivations, are in this week's homework write-up.

Coding is not graded, and you may use an AI assistant for it. The self-check cells
let you confirm your code is correct before you rely on its output. Run top to bottom,
make sure every figure and every PASS line appears, and submit the completed `.ipynb`.

In [ ]:
# --- setup ---------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## Part 1. Interpolating Runge's function on equispaced nodes

You might expect that using more interpolation nodes always gives a better fit.
Runge's function

$$ f(x) = \frac{1}{1+25x^2} $$

shows this is false. You will build the interpolant on equally spaced nodes and watch
the error as the degree grows.

In [ ]:
def equispaced_nodes(a, b, n):
    """Return the n+1 equally spaced nodes on [a, b], endpoints included.
    Input:  a, b floats with a < b; n an integer >= 0.
    Output: 1D numpy array of length n+1.
    """
    # TODO: your implementation
    raise NotImplementedError

In [ ]:
# provided helpers --------------------------------------------------------
def interpolant(nodes, values):
    """Return the polynomial interpolant through (nodes, values) as a callable p(x).
    Provided; uses the numerically stable barycentric formula so that the method of
    interpolation is not what this lab is about."""
    nodes = np.asarray(nodes, float); values = np.asarray(values, float)
    m = len(nodes); w = np.ones(m)
    for j in range(m):
        d = nodes[j] - nodes; d[j] = 1.0
        w[j] = 1.0 / np.prod(d)                 # barycentric weights
    def p(xq):
        xq = np.asarray(xq, float); out = np.empty_like(xq)
        flat, xf = out.ravel(), xq.ravel()
        for i, x in enumerate(xf):
            diff = x - nodes
            hit = np.where(diff == 0)[0]
            if hit.size:                        # x coincides with a node
                flat[i] = values[hit[0]]
            else:
                t = w / diff
                flat[i] = np.dot(t, values) / np.sum(t)
        return out
    return p

def max_error(f, p, a, b, N=2001):
    """Maximum of |f - p| on a fine grid over [a, b]."""
    xx = np.linspace(a, b, N)
    return np.max(np.abs(f(xx) - p(xx)))

In [ ]:
# === self-check (do not edit): every line must print PASS ===
_x = equispaced_nodes(-1, 1, 5)
assert len(_x) == 6 and np.isclose(_x[0], -1) and np.isclose(_x[-1], 1)
print("PASS  nodes")
_g = lambda x: 2*x**3 - x + 1
_p = interpolant(_x, _g(_x))
assert np.allclose(_p(_x), _g(_x), atol=1e-8)
print("PASS  interpolates the data at the nodes")
assert max_error(_g, _p, -1, 1) < 1e-9          # exact on a cubic
print("PASS  exact on a low-degree polynomial")

In [ ]:
# experiment: overlays and the error curve --------------------------------
def runge(x): return 1.0 / (1.0 + 25.0 * x**2)

xx = np.linspace(-1, 1, 800)
plt.figure(); plt.plot(xx, runge(xx), "k-", lw=2, label="f")
for n in (4, 10, 20):
    xn = equispaced_nodes(-1, 1, n)
    plt.plot(xx, interpolant(xn, runge(xn))(xx), label=f"p_{n}")
plt.ylim(-0.5, 1.5); plt.legend()
plt.title("Equispaced interpolation of Runge's function"); plt.show()

degs, errs = np.arange(0, 26), []
for n in degs:
    xn = equispaced_nodes(-1, 1, n)
    errs.append(max_error(runge, interpolant(xn, runge(xn)), -1, 1))
plt.figure(); plt.semilogy(degs, errs, "r.-")
plt.xlabel("degree n"); plt.ylabel("max error on [-1, 1]")
plt.title("Equispaced error vs degree"); plt.show()

## Part 2. Chebyshev nodes

The interpolation error formula from lecture is

$$ f(x) - p_n(x) = \frac{f^{(n+1)}(\xi)}{(n+1)!}\,\omega(x), \qquad \omega(x)=\prod_{k=0}^{n}(x-x_k). $$

The factor $f^{(n+1)}/(n+1)!$ is fixed by $f$, but $\omega$ depends only on where the
nodes sit, and Part 1 shows the trouble concentrates near the endpoints. The Chebyshev
nodes are placed to keep $\max_x|\omega(x)|$ small. On $[-1,1]$ they are

$$ x_j=\cos\frac{(2j+1)\pi}{2(n+1)},\qquad j=0,\dots,n, $$

obtained by spacing $n+1$ points evenly around the upper unit semicircle over $[-1,1]$
and projecting them onto the $x$-axis.

These nodes are tied to the Chebyshev polynomials $T_m(x)=\cos(m\arccos x)$, which
satisfy the recurrence $T_{m+1}(x)=2x\,T_m(x)-T_{m-1}(x)$ with $T_0=1,\ T_1=x$ and are
therefore polynomials of degree $m$. The nodes $x_j$ above are exactly the roots of
$T_{n+1}$.

> **Theorem (Chebyshev minimax).** Among all monic polynomials of degree $n+1$ on
> $[-1,1]$, the one with the smallest maximum absolute value is $2^{-n}T_{n+1}$, with
> $\max_{[-1,1]}|2^{-n}T_{n+1}|=2^{-n}$. Taking the nodes to be the roots of $T_{n+1}$
> makes the monic node polynomial $\omega=2^{-n}T_{n+1}$, so $\max_{[-1,1]}|\omega|=2^{-n}$,
> the smallest possible.

The derivations of the node formula, the degree of $T_m$, the root property, and this
minimax bound are assigned in this week's homework write-up. You verify the bound
numerically in Part 3.

In [ ]:
def chebyshev_nodes(a, b, n):
    """Return the n+1 Chebyshev nodes on [a, b]. On [-1,1] they are
    cos((2j+1)*pi/(2(n+1))) for j=0..n; map affinely to [a,b].
    Output: 1D numpy array of length n+1 (any order is fine).
    """
    # TODO: build the [-1,1] nodes with the cosine formula, then map to [a,b]
    raise NotImplementedError

In [ ]:
# === self-check (do not edit) ===
_c = chebyshev_nodes(-1, 1, 7)
assert len(_c) == 8
assert np.all(_c > -1 - 1e-12) and np.all(_c < 1 + 1e-12)
assert np.allclose(np.cos(8 * np.arccos(np.clip(_c, -1, 1))), 0, atol=1e-10)
print("PASS  Chebyshev nodes are the roots of T_(n+1)")

## Part 3. Chebyshev interpolation and the node polynomial

Repeat the Runge experiment with Chebyshev nodes, and inspect $\omega(x)$ directly to
see what changed.

In [ ]:
# Chebyshev interpolation of Runge, and both error curves overlaid ---------
xx = np.linspace(-1, 1, 800)
plt.figure(); plt.plot(xx, runge(xx), "k-", lw=2, label="f")
for n in (4, 10, 20):
    xn = chebyshev_nodes(-1, 1, n)
    plt.plot(xx, interpolant(xn, runge(xn))(xx), label=f"q_{n}")
plt.ylim(-0.5, 1.5); plt.legend()
plt.title("Chebyshev interpolation of Runge's function"); plt.show()

degs = np.arange(0, 26)
err_eq, err_ch = [], []
for n in degs:
    xe = equispaced_nodes(-1, 1, n); err_eq.append(max_error(runge, interpolant(xe, runge(xe)), -1, 1))
    xc = chebyshev_nodes(-1, 1, n);  err_ch.append(max_error(runge, interpolant(xc, runge(xc)), -1, 1))
plt.figure()
plt.semilogy(degs, err_eq, "r.-", label="equispaced")
plt.semilogy(degs, err_ch, "b.-", label="Chebyshev")
plt.xlabel("degree n"); plt.ylabel("max error on [-1, 1]"); plt.legend()
plt.title("Equispaced vs Chebyshev error"); plt.show()

In [ ]:
def omega(x, nodes):
    """Evaluate the node polynomial omega(x) = prod_k (x - nodes[k]).
    x may be a scalar or a numpy array; return the same shape.
    """
    # TODO: form the product over the nodes
    raise NotImplementedError

In [ ]:
# === self-check (do not edit) ===
_nd = equispaced_nodes(-1, 1, 6)
assert np.allclose(omega(_nd, _nd), 0.0, atol=1e-12)        # vanishes at its nodes
assert np.isclose(omega(0.3, [0.0, 1.0]), 0.3 * (0.3 - 1.0))
print("PASS  omega")

xx = np.linspace(-1, 1, 4001)
for name, nd in [("equispaced", equispaced_nodes(-1,1,20)), ("Chebyshev", chebyshev_nodes(-1,1,20))]:
    print(f"{name:>11}: max|omega| = {np.max(np.abs(omega(xx, nd))):.3e}")
print(f"{'':>11}  (Chebyshev target 2^-20 = {2.0**-20:.3e})")
plt.figure()
plt.semilogy(xx, np.abs(omega(xx, equispaced_nodes(-1,1,20))) + 1e-30, "r-", label="equispaced")
plt.semilogy(xx, np.abs(omega(xx, chebyshev_nodes(-1,1,20))) + 1e-30, "b-", label="Chebyshev")
plt.xlabel("x"); plt.ylabel("|omega(x)|,  n = 20"); plt.legend()
plt.title("Node polynomial for the two node sets"); plt.show()

## Where this goes next

Node placement is a design choice, and the node polynomial $\omega$ is the quantity it
controls. The same minimax idea returns when we choose quadrature nodes later in the
course. The written questions for this lab, including the Part 2 derivations, are in
this week's homework write-up, and later homework assumes you have run this lab.